# Extension 1 — Dynamic misspecification: VARMA(4,1) DGP (Ch. 4)

Stress-tests the Ch. 3 baseline ranking under **dynamic misspecification**. The DGP augments the baseline with a first-order moving-average term,
$$ y_t = \sum_{i=1}^4 A_i y_{t-i} + u_t + \Theta u_{t-1}, \qquad \Theta=\theta I_2,\quad u_t=B\varepsilon_t,\quad \varepsilon_t\sim\mathcal N(0,I_2), $$
with misspecification strength $\theta\in\{0.1,0.3,0.6\}$ ($\theta=0$ nests the baseline VAR(4)). The lag matrices $A_i$ and impact matrix $B$ are inherited from the baseline, so **all** variation is the MA component.

The VARMA is invertible, so it admits a VAR($\infty$) representation: a high-order VAR($q$) can approximate it, the equal-lag **VAR(4)** cannot (it carries an asymptotic bias that does not vanish with $T$), and **LP(4)** is robust by construction \[Montiel Olea et al.\]. Estimators (fixed **LP(4)** and the **VAR($q$) sweep**, $q=1\ldots p+H-1$) are applied **unchanged**; the estimand is the true VARMA structural IRF $\theta_h=(\Phi_h B)[1,1]$, computed analytically per $\theta$.

In [ ]:
from functools import partial

import numpy as np
import matplotlib.pyplot as plt

from mcsim.dgp import (VARSpec, VARMASpec, simulate_varma, varma_irf,
                       scale_to_persistence, spectral_radius)
from mcsim.estimators import estimate_lp_irf, estimate_var_irf
from mcsim.simulation import MCConfig, run

## 0. Set up model parameters

In [ ]:
P, H, RHO = 4, 20, 0.95          # Rho in {0.5, 0.7, 0.95}; run the file once per persistence
T = [100, 200, 240, 500]
THETAS = [0.0, 0.1, 0.3, 0.6]    # MA misspecification strength (theta=0 nests baseline VAR(4))
T_SHOW = 240                     # sample size used for the per-theta diagnostic panels

M  = np.array([[0.50, 0.10], [0.10, 0.50]])      # symmetric, positive -> real, decaying
A0 = np.array([M, 0.4 * M, 0.2 * M, 0.1 * M])     # VAR(4): geometrically decaying lag matrices
B  = np.array([[1.0, 0.0], [0.5, 1.0]])           # recursive (lower-triangular) impact matrix
A_RHO = scale_to_persistence(A0, RHO)             # AR part scaled to target persistence; shared across theta

# VARMA(4,1) with Theta_1 = theta * I_2; theta=0 reproduces the baseline VAR(4) exactly.
DGPS = {th: VARMASpec(A=A_RHO, Theta=np.array([th * np.eye(2)]), B=B) for th in THETAS}
print("AR spectral radius (persistence):", round(spectral_radius(A_RHO), 3),
      "| MA(1) root modulus 1/theta > 1 for theta<1 -> invertible (VAR-infinity exists)")

N_REPS = 5000  # nr of MC reps

## 1. DGP — how the MA term distorts the true IRF

Same AR dynamics and recursive identification as the baseline; the MA term $\Theta u_{t-1}$ injects extra **short-horizon** propagation, then the AR dynamics take over. Larger $\theta$ lifts the early-horizon estimand $\theta_h=(\Phi_h B)[1,1]$; $\theta=0$ recovers the baseline VAR(4) IRF.

In [ ]:
# True structural IRFs (no estimation) for each misspecification strength
hgrid = np.arange(H + 1)
truth_byTheta = {th: varma_irf(DGPS[th], H) for th in THETAS}

cth = {th: plt.cm.viridis(i / max(len(THETAS) - 1, 1)) for i, th in enumerate(THETAS)}
fig, ax = plt.subplots(figsize=(8, 4.5))
for th in THETAS:
    lab = "θ=0  (baseline VAR(4))" if th == 0 else f"θ={th}"
    ax.plot(hgrid, truth_byTheta[th], "o-", lw=2.2, color=cth[th], label=lab)
ax.axhline(0, color="k", lw=0.5)
ax.set_title(f"True structural IRF  θ_h = (Φ_h B)[1,1]  across MA strength — ρ={RHO}")
ax.set_xlabel("horizon $h$"); ax.set_ylabel(r"$\theta_h$"); ax.legend(title="misspecification")
fig.tight_layout(); plt.show()

## 2. Monte Carlo — LP(4) + VAR(q) sweep × misspecification × sample size

The three-way design of Ch. 4: for each $\theta\in\{0,0.1,0.3,0.6\}$ and $T\in\{100,200,240,500\}$ we run the fixed **LP(4)** and the full **VAR(q) sweep** ($q=1\ldots23$) on VARMA-generated data, **without modification**. The estimand is $T$-independent but **$\theta$-dependent** (`truth_byTheta`). Raw stacks land in `results[(θ,T)]` and horizon-wise RMSE in `rmse[(θ,T)]`.

In [ ]:
VAR_ORDERS = list(range(1, P + H))                  # 1, 2, ..., 22, 23

estimators = {"LP(4)": partial(estimate_lp_irf, p=P, horizon=H)}
estimators.update({f"VAR({q})": partial(estimate_var_irf, p=q, horizon=H) for q in VAR_ORDERS})


def varma_dgp(rng, Tlen, spec):
    return simulate_varma(spec, Tlen, rng)


results, rmse = {}, {}                               # keyed by (theta, T)
for th in THETAS:
    truth = truth_byTheta[th]                        # estimand depends on theta
    for Tval in T:
        cfg = MCConfig(n_reps=N_REPS, T=Tval, horizon=H, seed=42, n_jobs=-1,
                       progress=True, estimators=estimators)
        res = run(partial(varma_dgp, spec=DGPS[th]), cfg)
        results[(th, Tval)] = res
        rmse[(th, Tval)] = {nm: np.sqrt(np.nanmean((st - truth) ** 2, axis=0))
                            for nm, st in res["irfs"].items()}
        fails = {k: v for k, v in res["n_failures"].items() if v}
        print(f"theta={th:<4} T={Tval:>4}: done  |  failures: {fails or 'none'}")

## 3. RMSE comparison: does the ranking survive misspecification?

The **complexity frontier** (horizon-averaged RMSE vs VAR order) at $T=240$, one curve per $\theta$, with LP(4) as a same-colour dashed reference. As $\theta$ grows the equal-lag (VAR(4)) end of the frontier should lift — four lags cannot fit the MA dynamics — while higher-order VARs and LP(4) absorb it. The table reports every $(\theta,T)$ cell.

In [ ]:
# (1) complexity frontier per theta, at T_SHOW
fig, ax = plt.subplots(figsize=(9, 5.5))
for th in THETAS:
    rm = rmse[(th, T_SHOW)]
    avg_var = [np.mean(rm[f"VAR({q})"]) for q in VAR_ORDERS]
    lab = "θ=0 (baseline)" if th == 0 else f"θ={th}"
    ax.plot(VAR_ORDERS, avg_var, "o-", ms=3, color=cth[th], label=lab)
    ax.axhline(np.mean(rm["LP(4)"]), ls="--", color=cth[th], alpha=0.7)
ax.axvline(4, color="gray", ls=":", lw=1)
ax.set_title(f"Complexity frontier across θ  (T={T_SHOW};  solid = VAR(q), dashed = LP(4), same colour)")
ax.set_xlabel("VAR order q"); ax.set_ylabel("horizon-averaged RMSE")
ax.legend(fontsize=8, title="misspecification")
fig.tight_layout(); plt.show()

# (2) full (theta, T) summary table
print(f"{'theta':>6} {'T':>5} {'LP(4)':>8} {'VAR(4)':>8} {'bestVAR':>14} {'LP-VAR4':>9}")
print("-" * 56)
for th in THETAS:
    for Tval in T:
        rm = rmse[(th, Tval)]
        lp = float(np.mean(rm["LP(4)"])); v4 = float(np.mean(rm["VAR(4)"]))
        avgs = {q: float(np.mean(rm[f"VAR({q})"])) for q in VAR_ORDERS}
        bq = min(avgs, key=avgs.get)
        print(f"{th:>6} {Tval:>5} {lp:8.3f} {v4:8.3f} {f'VAR({bq})={avgs[bq]:.3f}':>14} {lp - v4:+9.3f}")
print("(horizon-averaged RMSE;  LP-VAR4 > 0 = VAR(4) better)")

### RMSE by horizon, per misspecification strength

One panel per $\theta$ at $T=240$: the VAR sweep (dark = low order → yellow = high) with LP(4) and the equal-lag VAR(4) highlighted. Misspecification bias concentrates at short horizons (where $\Theta u_{t-1}$ acts), so watch VAR(4) lift there as $\theta$ grows while LP(4) stays flat.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for ax, th in zip(axes.ravel(), THETAS):
    rm = rmse[(th, T_SHOW)]
    for i, q in enumerate(VAR_ORDERS):
        ax.plot(hgrid, rm[f"VAR({q})"], color=plt.cm.viridis(i / len(VAR_ORDERS)), lw=1, alpha=0.6)
    ax.plot(hgrid, rm["VAR(4)"], "k--", lw=2, label="VAR(4)")
    ax.plot(hgrid, rm["LP(4)"], color="crimson", lw=2.6, label="LP(4)")
    ttl = "θ=0 (baseline)" if th == 0 else f"θ={th}"
    ax.set_title(ttl); ax.set_xlabel("horizon $h$"); ax.set_ylabel("RMSE"); ax.legend(fontsize=7)
fig.suptitle(f"RMSE by horizon across θ  (T={T_SHOW};  VAR sweep: dark = low order -> yellow = high)", fontweight="bold")
fig.tight_layout(); plt.show()

## 4. The misspecification channel — bias of VAR(4) vs robustness of LP(4)

The central Ch. 4 mechanism. The equal-lag **VAR(4)** cannot represent the VARMA(4,1) with four lags, so it carries an **asymptotic bias** that does not vanish with $T$ — concentrated at the short horizons where $\Theta u_{t-1}$ acts. **LP(4)** is robust by construction, so its bias stays ≈ 0. Below: bias $=\overline{\hat\theta_h}-\theta_h$ at $T=240$, one curve per $\theta$.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for nm, ax in [("VAR(4)", a1), ("LP(4)", a2)]:
    for th in THETAS:
        stack = results[(th, T_SHOW)]["irfs"][nm]
        bias = np.nanmean(stack, axis=0) - truth_byTheta[th]
        lab = "θ=0" if th == 0 else f"θ={th}"
        ax.plot(hgrid, bias, "o-", ms=3, color=cth[th], label=lab)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(nm); ax.set_xlabel("horizon $h$")
    ax.set_ylabel(r"bias $=\overline{\hat\theta_h}-\theta_h$"); ax.legend(fontsize=8, title="θ")
fig.suptitle(f"Misspecification bias by horizon — T={T_SHOW}  "
             f"(VAR(4) biased & growing in θ;  LP(4) ≈ unbiased)", fontweight="bold")
fig.tight_layout(); plt.show()

### Predicted IRF vs the truth (levels, not RMSE)

At $T=240$, one panel per $\theta$: the true VARMA IRF (dashed black), and the **mean** estimated IRF across replications for VAR(4) and LP(4) with central 90% bands (5th–95th pct). The gap between a solid mean line and the dashed truth is *bias*; band width is *dispersion*. As $\theta$ grows VAR(4)'s mean peels away from the truth at short horizons while LP(4) tracks it.

In [ ]:
selp = ["VAR(4)", "LP(4)"]
colp = {"VAR(4)": "black", "LP(4)": "crimson"}

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for ax, th in zip(axes.ravel(), THETAS):
    tr = truth_byTheta[th]
    for nm in selp:
        stack = results[(th, T_SHOW)]["irfs"][nm]
        mean = np.nanmean(stack, axis=0)
        lo, hi = np.nanpercentile(stack, [5, 95], axis=0)
        ax.fill_between(hgrid, lo, hi, color=colp[nm], alpha=0.15)
        ax.plot(hgrid, mean, color=colp[nm], lw=2.2, label=f"mean {nm}")
    ax.plot(hgrid, tr, "k--", lw=2.0, label="true IRF")
    ax.axhline(0, color="gray", lw=0.5)
    ttl = "θ=0 (baseline)" if th == 0 else f"θ={th}"
    ax.set_title(ttl); ax.set_xlabel("horizon $h$"); ax.set_ylabel(r"$\theta_h$"); ax.legend(fontsize=7)
fig.suptitle(f"Predicted IRF vs truth across θ — T={T_SHOW}, B={N_REPS}  (band = central 90%)",
             fontweight="bold")
fig.tight_layout(); plt.show()

## 5. Point-estimation metrics under misspecification (Ch. 3 metrics)

Same per-horizon **bias / variance / MSE / RMSE** decomposition as the baseline, now at the strongest misspecification $\theta=0.6$, $T=240$. Unlike the correctly-specified baseline (VAR bias ≈ 0), here low-order VARs carry genuine bias, so $\mathrm{MSE}_h=\mathrm{Bias}_h^2+\mathrm{Var}_h$ has a non-negligible squared-bias component at short horizons. $\mathrm{MCSE}(\widehat{\mathrm{Bias}}_h)=\sqrt{\widehat{\mathrm{Var}}_h/B}$ bounds simulation noise.

In [ ]:
import pandas as pd
from IPython.display import display


def metrics(stack, truth):
    """Per-horizon Ch. 3 point metrics for a (B, H+1) stack of IRF estimates."""
    B = int(np.sum(~np.isnan(stack[:, 0])))
    mean = np.nanmean(stack, axis=0)
    bias = mean - truth
    var = np.nanvar(stack, axis=0, ddof=1)
    mse = np.nanmean((stack - truth) ** 2, axis=0)
    return dict(bias=bias, var=var, mse=mse, rmse=np.sqrt(mse),
                mcse_bias=np.sqrt(var / B), B=B)


TH_SHOW, Tval = 0.6, T_SHOW                          # strongest misspecification cell
truth = truth_byTheta[TH_SHOW]
sel = ["LP(4)", "VAR(2)", "VAR(4)", "VAR(8)", "VAR(16)", "VAR(23)"]
colm = {"LP(4)": "crimson", "VAR(2)": "tab:blue", "VAR(4)": "black",
        "VAR(8)": "tab:green", "VAR(16)": "darkgreen", "VAR(23)": "tab:purple"}
m = {nm: metrics(results[(TH_SHOW, Tval)]["irfs"][nm], truth) for nm in sel}

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for nm in sel:
    ax[0, 0].plot(hgrid, m[nm]["bias"], color=colm[nm], lw=1.8, label=nm)
    ax[0, 1].plot(hgrid, m[nm]["var"], color=colm[nm], lw=1.8, label=nm)
    ax[1, 0].plot(hgrid, m[nm]["mse"], color=colm[nm], lw=1.8, label=nm)
    ax[1, 1].plot(hgrid, m[nm]["rmse"], color=colm[nm], lw=1.8, label=nm)
for nm in ("LP(4)", "VAR(4)"):                       # MCSE band on bias (sim. uncertainty)
    lo, hi = m[nm]["bias"] - 1.96 * m[nm]["mcse_bias"], m[nm]["bias"] + 1.96 * m[nm]["mcse_bias"]
    ax[0, 0].fill_between(hgrid, lo, hi, color=colm[nm], alpha=0.15)
ax[0, 0].axhline(0, color="k", lw=0.5)
ax[0, 0].set_title("Bias  (shaded = ±1.96·MCSE for LP/VAR4)")
ax[0, 1].set_title("Variance"); ax[1, 0].set_title("MSE"); ax[1, 1].set_title("RMSE")
for a in ax.ravel():
    a.set_xlabel("horizon $h$"); a.legend(fontsize=7)
fig.suptitle(f"Ch. 3 point metrics under misspecification — θ={TH_SHOW}, T={Tval}, B={N_REPS}",
             fontweight="bold")
fig.tight_layout(); plt.show()

# compact table: bias and RMSE at selected horizons
report = [1, 8, 20]
rows = {nm: {(f"h={h}", k): m[nm][{"bias": "bias", "RMSE": "rmse"}[k]][h]
             for h in report for k in ("bias", "RMSE")} for nm in sel}
tab = pd.DataFrame(rows).T
tab.columns = pd.MultiIndex.from_tuples(tab.columns)
print(f"θ={TH_SHOW}, T={Tval}, B={N_REPS}   |   typical MCSE(bias) ~ "
      f"{np.mean([m['VAR(4)']['mcse_bias'][h] for h in report]):.4f}")
display(tab.round(4))

## 6. Inference — coverage under misspecification (Ch. 3)

Coverage of nominal 95% CIs for the **true** VARMA $\theta_h$. The CI machinery is unchanged from the baseline:

* **VAR(q): delta method** — propagates reduced-form coefficient uncertainty through $\theta_h=\hat\Psi_h\hat B e_1$ (variance only; $B$ held fixed).
* **LP(4): HC1-robust SE** of the shock coefficient.

Under misspecification the delta-method SE still captures *variance*, but the VAR(4) point estimate is *biased*; a variance-only CI centred on a biased estimate **systematically misses** the truth, so VAR(4) coverage should fall as $\theta$ grows while LP(4) (≈ unbiased) stays near nominal. The validation panel (delta SE vs empirical SD, at $\theta=0.6$) confirms the SE itself is still correct — so any coverage loss is attributable to bias, not a broken SE.

In [140]:
from mcsim.dgp import var_ma_matrices
from mcsim.estimators import fit_var_ols


def var_theta_se(y, q, horizon, shock=0, response=0):
    """Structural IRF point estimate AND delta-method standard error for a VAR(q).

    Returns (theta_hat, se) each length horizon+1, for the response of variable
    `response` to structural shock `shock`.  The SE propagates the sampling
    uncertainty of the reduced-form OLS coefficients A through the nonlinear map
        theta_h = e_response' . Psi_h(A) . B . e_shock ,
    holding the impact matrix B = chol(Sigma_u) FIXED (Ch. 3's choice: only the
    reduced-form coefficients are propagated).
    """
    # --- 1. Reduced-form OLS fit -------------------------------------------------
    # A_hat: (q, k, k) lag matrices;  Sig: (k, k) residual covariance.
    A, Sig, _resid = fit_var_ols(y, q)
    k = A.shape[1]

    # --- 2. Structural objects (B held fixed for the delta method) ---------------
    Bm = np.linalg.cholesky(Sig)            # recursive impact matrix  B = chol(Sigma_u)
    c = Bm[:, shock]                        # column = impact vector of the chosen shock, B e_shock
    er = np.zeros(k); er[response] = 1.0    # selector e_response (picks the response variable)

    # Reduced-form MA matrices Psi_0..Psi_H and the point IRF theta_h = er' Psi_h c.
    Psi = var_ma_matrices(A, horizon)       # (H+1, k, k)
    theta = np.array([er @ Psi[h] @ c for h in range(horizon + 1)])

    # --- 3. OLS coefficient covariance  Cov(vec C) = Sigma_u (x) (X'X)^{-1} -------
    # Rebuild the regressor matrix X_t = [1, y_{t-1}, ..., y_{t-q}] (same as fit_var_ols).
    T = y.shape[0]; rows = T - q
    X = np.ones((rows, 1 + q * k))
    for i in range(1, q + 1):
        X[:, 1 + (i - 1) * k: 1 + i * k] = y[q - i: T - i]
    XtXi = np.linalg.inv(X.T @ X)           # (1+qk, 1+qk)

    # Parameters we propagate: every entry A_i[a, b].  Index convention matches
    # fit_var_ols, where A_i[a, b] = coef[row, a] with row = 1 + i*k + b
    # (i is 0-based lag, a = equation/response row, b = which variable's lag).
    params = [(i, a, b) for i in range(q) for a in range(k) for b in range(k)]

    # Sigma_alpha[p, p'] = Cov(A_i[a,b], A_{i'}[a',b']) = Sigma_u[a,a'] * (X'X)^{-1}[row, row'].
    # (From Cov(vec C) = Sigma_u (x) (X'X)^{-1}: equations couple via Sigma_u,
    #  regressors via (X'X)^{-1}.)
    Sig_a = np.array([[Sig[a, a2] * XtXi[1 + i * k + b, 1 + i2 * k + b2]
                       for (i2, a2, b2) in params] for (i, a, b) in params])

    # --- 4. Gradient d_h = d theta_h / d alpha via the differentiated MA recursion
    # Psi_h = sum_{j=1}^{min(h,q)} A_j Psi_{h-j}.  Differentiating w.r.t. the single
    # entry A_{i+1}[a,b] gives, with E_{ab} the unit matrix (1 at (a,b)):
    #   dPsi_h = sum_j A_j dPsi_{h-j}  +  E_{ab} Psi_{h-i-1}   (the +E term only when j=i+1)
    Em = {}
    for a in range(k):
        for b in range(k):
            mat = np.zeros((k, k)); mat[a, b] = 1.0; Em[(a, b)] = mat

    dPsi = [{p: np.zeros((k, k)) for p in params}]     # dPsi at h=0 is all zeros
    for h in range(1, horizon + 1):
        cur = {}
        for p in params:
            i, a, b = p
            acc = np.zeros((k, k))
            for j in range(1, min(h, q) + 1):
                acc = acc + A[j - 1] @ dPsi[h - j][p]       # chain-rule term
                if j - 1 == i:                              # direct term: this lag IS the parameter
                    acc = acc + Em[(a, b)] @ Psi[h - j]
            cur[p] = acc
        dPsi.append(cur)

    # --- 5. Var(theta_h) = d_h' Sigma_alpha d_h, with d_h[p] = er' dPsi_h[p] c ----
    se = np.zeros(horizon + 1)
    for h in range(horizon + 1):
        d = np.array([er @ dPsi[h][p] @ c for p in params])
        se[h] = np.sqrt(max(d @ Sig_a @ d, 0.0))           # clip tiny negatives from roundoff
    return theta, se


def lp_theta_se(y, p, horizon, shock=0, response=0):
    """LP(p) point IRF AND heteroskedasticity-robust (HC1) standard error per horizon.

    The shock is recovered from a first-stage VAR(p) (same as estimate_lp_irf);
    at each horizon h the IRF is the OLS slope on the shock x_t in
        y_{response, t+h} = mu + beta_h x_t + sum_l delta_l' y_{t-l} + xi,
    and its SE is the White/HC1 robust SE of that slope.  HC (not HAC) is used
    because the structural shock is a martingale difference (Plagborg-Moller & Wolf);
    note LP residuals are serially correlated for h>0, so this can mildly under-state
    uncertainty at long horizons.
    """
    y = np.asarray(y, dtype=float)
    T, k = y.shape

    # First-stage VAR(p): residuals -> structural shock  x_t = e_shock' B^{-1} u_t.
    A, Sig, resid = fit_var_ols(y, p)
    Bm = np.linalg.cholesky(Sig)
    eps = np.linalg.solve(Bm, resid.T).T          # structural innovations (rows aligned t=p..T-1)
    x = eps[:, shock]

    rows = T - p
    lags = np.empty((rows, p * k))                # controls [y_{t-1}, ..., y_{t-p}]
    for i in range(1, p + 1):
        lags[:, (i - 1) * k: i * k] = y[p - i: T - i]

    theta = np.full(horizon + 1, np.nan)
    se = np.full(horizon + 1, np.nan)
    for h in range(horizon + 1):
        m = rows - h                              # usable base times at this horizon
        if m <= 0:
            break
        dep = y[p + h: p + h + m, response]        # y_{response, t+h}
        Xr = np.column_stack([np.ones(m), x[:m], lags[:m]])   # [const, shock, lags]
        beta, *_ = np.linalg.lstsq(Xr, dep, rcond=None)
        e = dep - Xr @ beta                       # residuals
        XtXi = np.linalg.inv(Xr.T @ Xr)
        # HC1 robust covariance:  (X'X)^{-1} (X' diag(e^2) X) (X'X)^{-1} * m/(m-kreg)
        meat = Xr.T @ (Xr * (e ** 2)[:, None])
        V = XtXi @ meat @ XtXi * (m / (m - Xr.shape[1]))
        theta[h] = beta[1]                        # slope on the shock (col index 1)
        se[h] = np.sqrt(V[1, 1])
    return theta, se

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm

T_COV = T_SHOW         # one sample-size cell (Ch. 3 would sweep T as in section 2)
N_COV = 5000           # replications for the coverage MC
Z = 1.96               # normal critical value for a nominal 95% two-sided CI
EST_COV = ["VAR(4)", "LP(4)"]   # headline pair: misspecified equal-lag VAR vs robust LP


def _coverage_rep(seed, spec):
    """One replication: simulate from the VARMA spec, return {estimator: (theta_hat, se)}."""
    rng = np.random.default_rng(seed)
    y = simulate_varma(spec, T_COV, rng)
    with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
        return {"VAR(4)": var_theta_se(y, 4, H),
                "LP(4)":  lp_theta_se(y, 4, H)}


coverage = {}          # theta -> {estimator: coverage_h}
val = {}               # SE-vs-SD validation at the strongest theta
for th in THETAS:
    tr = truth_byTheta[th]
    # Independent, reproducible RNG stream per replication (parallel-safe), as in mcsim.run.
    seeds = np.random.SeedSequence(42).spawn(N_COV)
    reps = Parallel(n_jobs=-1)(delayed(_coverage_rep)(s, DGPS[th])
                               for s in tqdm(seeds, desc=f"coverage MC θ={th}"))
    theta_s = {nm: np.array([r[nm][0] for r in reps]) for nm in EST_COV}
    se_s = {nm: np.array([r[nm][1] for r in reps]) for nm in EST_COV}
    # Coverage_h = fraction of CIs [theta_hat +/- Z*se] that contain the TRUE theta_h.
    coverage[th] = {nm: np.nanmean((theta_s[nm] - Z * se_s[nm] <= tr) &
                                   (tr <= theta_s[nm] + Z * se_s[nm]), axis=0) for nm in EST_COV}
    if th == 0.6:      # validation: delta-method SE should still match the empirical SD
        val = dict(emp=np.nanstd(theta_s["VAR(4)"], axis=0, ddof=1),
                   se=np.nanmean(se_s["VAR(4)"], axis=0))

# --- coverage per estimator (one curve per theta) + SE validation ----------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, nm in zip(axes[:2], EST_COV):
    for th in THETAS:
        lab = "θ=0" if th == 0 else f"θ={th}"
        ax.plot(hgrid, coverage[th][nm], "o-", ms=3, color=cth[th], label=lab)
    ax.axhline(0.95, color="gray", ls="--", label="nominal 0.95")
    ax.set_ylim(0, 1); ax.set_xlabel("horizon $h$"); ax.set_ylabel("coverage")
    ax.set_title(f"{nm} coverage across θ"); ax.legend(fontsize=7)

axv = axes[2]
axv.plot(hgrid, val["emp"], "o-", ms=3, label="empirical SD of $\\hat\\theta_h$")
axv.plot(hgrid, val["se"], "s--", ms=3, label="mean delta-method SE")
axv.set_xlabel("horizon $h$"); axv.set_ylabel("SE / SD")
axv.set_title("VAR(4) SE vs SD (θ=0.6): SE still valid"); axv.legend(fontsize=8)
fig.suptitle(f"Coverage of 95% CIs under misspecification — T={T_COV}, B={N_COV}", fontweight="bold")
fig.tight_layout(); plt.show()

print("horizon-averaged coverage:")
for nm in EST_COV:
    print(f"  {nm:>7}: " + "   ".join(f"θ={th}:{np.nanmean(coverage[th][nm]):.3f}" for th in THETAS))